# Silver Layer: Clean and Transform NSE Data
This notebook reads Bronze Delta data, validates the raw rows, applies cleanup logic, and writes the cleaned dataset into the Silver Delta layer.

In [ ]:
import logging
import yaml
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit, to_date

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("silver_transform")

## Load Configuration
Load the Silver configuration file and storage paths.

In [ ]:
# Determine config path relative to notebook location in Databricks
try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    config_path = Path(notebook_path).parent.parent / "configs" / "bronze_config.yaml"
except Exception:
    config_path = Path("../configs/bronze_config.yaml")

with open(str(config_path), "r") as f:
    config = yaml.safe_load(f)

storage_config = config["storage"]
bronze_path = storage_config["bronze_path"]
silver_path = "/mnt/stockdata/silver/nse_stock_processed"
bronze_format = storage_config.get("format", "delta")
partition_col = storage_config.get("partition_by", "ingestion_date")

logger.info(f"Bronze path: {bronze_path}")
logger.info(f"Silver path: {silver_path}")

## Initialize Spark Session
Create or attach to a Spark session configured for Delta Lake.

In [ ]:
spark = SparkSession.builder \
    .appName("Silver_NSE_Transform") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
logger.info(f"Spark session initialized: {spark.sparkContext.appName}")

## Read Bronze Data
Read the raw Bronze Delta table or path and review the schema.

In [ ]:
bronze_table = "stock_market.bronze_nse_stock_raw"

try:
    df_bronze = spark.table(bronze_table)
    logger.info(f"Loaded Bronze table {bronze_table}")
except Exception:
    df_bronze = spark.read.format(bronze_format).load(bronze_path)
    logger.info(f"Loaded Bronze path {bronze_path}")

df_bronze.printSchema()
df_bronze.show(5, truncate=False)

## Data Validation
Validate required columns, nulls, and basic OHLC integrity.

In [ ]:
required_columns = ["symbol", "open", "high", "low", "close", "tradedQuantity", "tradedValue", "ingestion_date"]
critical_columns = ["symbol", "close", "high", "low", "tradedQuantity"]

missing = [c for c in required_columns if c not in df_bronze.columns]
if missing:
    raise ValueError(f"Missing required Bronze columns: {missing}")

null_counts = {c: df_bronze.filter(col(c).isNull()).count() for c in critical_columns if c in df_bronze.columns}
null_summary = {c: n for c, n in null_counts.items() if n > 0}
if null_summary:
    raise ValueError(f"Null values present in critical columns: {null_summary}")

validation_df = df_bronze.filter((col("low") > col("close")) | (col("close") > col("high")))
invalid_count = validation_df.count()
if invalid_count > 0:
    raise ValueError(f"Found {invalid_count} rows with invalid OHLC values")

logger.info("Bronze validation passed")

## Clean and Transform
Drop raw-only columns, keep governance fields, and prepare the Silver schema.

In [ ]:
columns_to_drop = [
    "chart30dPath",
    "chart365dPath",
    "chartTodayPath",
    "date30dAgo",
    "date365dAgo",
    "perChange30d",
    "perChange365d",
    "meta_debtSeries",
    "meta_quotepreopenstatus_QuotePreOpenFlag",
    "meta_quotepreopenstatus_equityTime",
    "meta_quotepreopenstatus_preOpenTime",
    "meta_symbol",
    "nearWKH",
    "nearWKL",
    "meta_slb_isin"
]

clean_df = df_bronze.drop(*[c for c in columns_to_drop if c in df_bronze.columns])
clean_df = clean_df.withColumn("processed_timestamp", current_timestamp())
clean_df = clean_df.withColumn(partition_col, to_date(col(partition_col)))

clean_df.printSchema()
clean_df.show(5, truncate=False)

## Write Silver Delta
Write the cleaned dataset to the Silver Delta path and register the table.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS stock_market")

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy(partition_col) \
    .option("overwriteSchema", "true") \
    .save(silver_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS stock_market.silver_nse_stock_processed USING DELTA LOCATION '{silver_path}'")

logger.info(f"Written Silver Delta table to {silver_path}")
display(clean_df.limit(5))
